# C1 — Predictive Coding Unified Network (PCUN)

Aynı tahmin hatası (ε) hem routing hem yerel ağırlık güncellemesi için. Backprop yerine yerel update (prototip).

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/C1_PCUN_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
PARAMS = {
    "method_name":   "C1_PCUN",
    "run_name":      "wt2_seed42",
    "seed":          42,
    "results_root":  "./results",

    "model_name":    "gpt2",
    "tokenizer_name": "gpt2",

    "dataset":       "wikitext2",
    "max_seq_len":   512,
    "batch_size":    16,
    "num_workers":   2,

    "learning_rate": 5e-5,
    "num_epochs":    3,
    "warmup_steps":  100,
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    "limit_train_batches": 100,
    "limit_eval_batches":  50,

    "log_every_n_steps":        25,
    "save_checkpoints":         False,
    "checkpoint_every_n_steps": 500,
    "keep_last_n_checkpoints":  3,

    # ─ Method-specific (PCUN)
    "pc_precision_init": 1.0,
    "pc_update_lr":      0.01,
    "pc_delta":          1e-6,
    "pc_use_local_only": False,    # True yapınca yerel update aktif
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method — Predictive Coding Unified Network ───────────────
class PCUnifiedAttention(BaseRouterAttention):
    def __init__(self, gpt2_config, method_params):
        super().__init__(gpt2_config, method_params)
        mp = method_params
        self.delta = float(mp.get("pc_delta", 1e-6))
        self.update_lr = float(mp.get("pc_update_lr", 0.01))
        self.use_local_only = bool(mp.get("pc_use_local_only", False))
        self.layer_predictor = nn.Linear(self.embed_dim, self.embed_dim)
        self.log_precision = nn.Parameter(torch.tensor(0.0))

    def _layer_error(self, hidden):
        return hidden - self.layer_predictor(hidden)

    def _attend(self, q, k, v, hidden_states, attention_mask):
        eps = self._layer_error(hidden_states)
        err_norm = eps.norm(dim=-1)
        score_per_j = 1.0 / (err_norm + self.delta)
        scores = score_per_j.unsqueeze(1).expand(-1, q.size(-2), -1)
        scores = scores.unsqueeze(1).expand(-1, self.num_heads, -1, -1)

        causal = self._causal_mask(q.size(-2), k.size(-2), q.device)
        scores = scores.masked_fill(~causal, float("-inf"))
        if attention_mask is not None:
            scores = scores + attention_mask

        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = torch.matmul(weights, v)

        if self.training and self.use_local_only:
            with torch.enable_grad():
                precision = torch.exp(self.log_precision)
                local_loss = 0.5 * precision * (eps ** 2).sum()
                params = list(self.layer_predictor.parameters()) + [self.log_precision]
                grads = torch.autograd.grad(local_loss, params, retain_graph=True, allow_unused=True)
                for p, g in zip(params, grads):
                    if g is not None:
                        p.data -= self.update_lr * g

        self._last_stats = {"err_mean": float(err_norm.mean().item())}
        return out, weights

print("PCUnifiedAttention tanımlandı.")

In [ ]:
def build_model_and_adapter(params):
    wrapper = GPT2Wrapper(params["model_name"])
    wrapper.inject_attention(PCUnifiedAttention, params)
    print(f"PCUN injected → {sum(p.numel() for p in wrapper.parameters()):,} params")
    return wrapper, None

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL: {result['final_metrics']['test_ppl']:.4f}")
print(f"Final test BPC: {result['final_metrics']['test_bpc']:.4f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")